# dpyr — a full tour

`dpyr` brings R's [dplyr](https://dplyr.tidyverse.org/) grammar of data manipulation to Python, on top of [polars](https://pola.rs/). You build a pipeline by piping a `DataFrame` through verbs with the `|` operator.

Column names can be referenced three ways, from most to least R-like:

1. **Bare names** (new) — `read_csv` injects column names as variables into the global namespace so you can write `sepal_length` directly, just like R dplyr.
2. **`df.cols` namespace** — `df.cols.sepal_length` or `df.cols['my column']` for special characters / Python keywords.
3. **Global `c`** — `c.sepal_length`, the original shorthand (still works).

This notebook demonstrates every verb dpyr provides and notes where the behaviour matches dplyr.


## Setup

Import the verbs and a couple of helpers.

In [1]:
from dpyr import (
    DataFrame, read_csv, c,
    select, filter, mutate, arrange, desc,
    head, tail, distinct, rename, count,
    group_by, ungroup, summarize, summarise,
    slice_sample, sample_n, pull,
    inner_join, left_join, right_join, full_join, semi_join, anti_join,
    n, lag, lead, row_number, min_rank, dense_rank,
)
import polars as pl

## The pipe and column references

### Bare-name injection (default)

`read_csv` injects each column name into the global namespace as a `pl.col(...)` expression, so you can reference columns as plain Python variables — the closest Python can get to R's non-standard evaluation.

Column names are **sanitized** before injection:
- spaces and hyphens → underscore (`'my col'` → `my_col`)
- other non-identifier characters stripped
- names starting with a digit prefixed with `_` (`'2fast'` → `_2fast`)

Columns that would overwrite an existing variable, or whose sanitized name is a Python keyword, are skipped and a warning is issued. Use `df.cols['name']` to access them.

### `df.cols` namespace

`df.cols.sepal_length` returns `pl.col('sepal_length')` with validation that the column exists. Bracket notation `df.cols['my column']` handles special characters and Python keywords. `dir(df.cols)` lists all columns for tab-completion.

### Global `c` (legacy)

`c.column_name` is the original shorthand — it still works and accepts any name without validation.

In [2]:
# read_csv injects column names into the global namespace by default.
# After this call, sepal_length, sepal_width, petal_length, petal_width,
# and variety are available as bare names.
df = read_csv("iris.csv")
df.head()

Now use bare column names — no `c.` prefix needed:


In [ ]:
# Bare-name style: closest to R dplyr
df | filter(sepal_length > 5.0) | select(variety, sepal_length) | head()

In [ ]:
# df.cols namespace: validated access, special chars, keywords
df.cols  # shows all columns with sanitized names

In [ ]:
# df.cols works in pipelines and handles edge cases
df | filter(df.cols.sepal_length > 5.0) | head(3)

Disable injection and use the `df.cols` namespace or legacy `c` instead:


In [ ]:
df2 = read_csv("iris.csv", inject=False)
# Use df.cols or c — no bare names injected
df2 | filter(df2.cols.variety == "Setosa") | head(3)

## `select` — pick columns
Equivalent to dplyr's `select()`.

In [3]:
df | select(c.sepal_length, c.sepal_width, c.variety) | head()

sepal_length,sepal_width,variety
f64,f64,str
5.1,3.5,"""Setosa"""
4.9,3.0,"""Setosa"""
4.7,3.2,"""Setosa"""
4.6,3.1,"""Setosa"""
5.0,3.6,"""Setosa"""


## `filter` — keep rows matching a condition

In [4]:
df | filter(c.variety == "Setosa") | head()

sepal_length,sepal_width,petal_length,petal_width,variety
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""Setosa"""
4.9,3.0,1.4,0.2,"""Setosa"""
4.7,3.2,1.3,0.2,"""Setosa"""
4.6,3.1,1.5,0.2,"""Setosa"""
5.0,3.6,1.4,0.2,"""Setosa"""


## `mutate` — add or change columns
Expressions are evaluated **sequentially**, just like dplyr, so a later expression can use a column created earlier in the same call.

In [5]:
df | mutate(
    sepal_ratio = c.sepal_length / c.sepal_width,
    sepal_ratio_pct = c.sepal_ratio * 100,
) | select(c.sepal_length, c.sepal_width, c.sepal_ratio, c.sepal_ratio_pct) | head()

sepal_length,sepal_width,sepal_ratio,sepal_ratio_pct
f64,f64,f64,f64
5.1,3.5,1.457143,145.714286
4.9,3.0,1.633333,163.333333
4.7,3.2,1.46875,146.875
4.6,3.1,1.483871,148.387097
5.0,3.6,1.388889,138.888889


## `arrange` — sort rows
Wrap a column in `desc()` to sort descending. Like dplyr, missing values always sort to the end.

In [6]:
df | arrange(desc(c.sepal_length)) | head()

sepal_length,sepal_width,petal_length,petal_width,variety
f64,f64,f64,f64,str
7.9,3.8,6.4,2.0,"""Virginica"""
7.7,3.8,6.7,2.2,"""Virginica"""
7.7,2.6,6.9,2.3,"""Virginica"""
7.7,2.8,6.7,2.0,"""Virginica"""
7.7,3.0,6.1,2.3,"""Virginica"""


## `head` / `tail` — first / last rows

In [7]:
df | tail(3)

sepal_length,sepal_width,petal_length,petal_width,variety
f64,f64,f64,f64,str
6.5,3.0,5.2,2.0,"""Virginica"""
6.2,3.4,5.4,2.3,"""Virginica"""
5.9,3.0,5.1,1.8,"""Virginica"""


## `distinct` — unique rows
With columns named, only those columns are kept (dplyr's default). Pass `keep_all=True` to keep every column.

In [8]:
df | distinct(c.variety)

variety
str
"""Setosa"""
"""Versicolor"""
"""Virginica"""


## `rename` — `rename(new = old)`

In [9]:
df | rename(species = c.variety) | head()

sepal_length,sepal_width,petal_length,petal_width,species
f64,f64,f64,f64,str
5.1,3.5,1.4,0.2,"""Setosa"""
4.9,3.0,1.4,0.2,"""Setosa"""
4.7,3.2,1.3,0.2,"""Setosa"""
4.6,3.1,1.5,0.2,"""Setosa"""
5.0,3.6,1.4,0.2,"""Setosa"""


## `count` — tally rows per group
Returns the grouping columns plus a count column named `n`.

In [10]:
df | count(c.variety)

variety,n
str,u32
"""Setosa""",50
"""Versicolor""",50
"""Virginica""",50


## `group_by` + `summarize` — split-apply-combine
`group_by` produces a grouped frame; `summarize` aggregates within each group. `n()` gives the group size.

In [11]:
df | group_by(c.variety) | summarize(
    mean_sepal_length = c.sepal_length.mean(),
    max_petal_length = c.petal_length.max(),
    rows = n(),
)

variety,mean_sepal_length,max_petal_length,rows
str,f64,f64,u32
"""Setosa""",5.006,1.9,50
"""Versicolor""",5.936,5.1,50
"""Virginica""",6.588,6.9,50


### Grouped `mutate` (window functions)
Inside a grouped frame, `mutate` computes each expression **within** its group and keeps the grouping. Here every row gets its variety's mean sepal length, then we flag the above-average rows.

In [12]:
df \
    | group_by(c.variety) \
    | mutate(variety_mean = c.sepal_length.mean()) \
    | filter(c.sepal_length > c.variety_mean) \
    | ungroup() \
    | select(c.variety, c.sepal_length, c.variety_mean) \
    | head()

variety,sepal_length,variety_mean
str,f64,f64
"""Setosa""",5.1,5.006
"""Setosa""",5.4,5.006
"""Setosa""",5.4,5.006
"""Setosa""",5.8,5.006
"""Setosa""",5.7,5.006


### Ranking within groups
`row_number()`, `min_rank()` and `dense_rank()` restart in each group.

In [13]:
# row_number ranks ascending, so negate to rank the largest first.
df \
    | group_by(c.variety) \
    | mutate(rank = row_number(-c.sepal_length)) \
    | filter(c.rank <= 2) \
    | ungroup() \
    | arrange(c.variety, c.rank) \
    | select(c.variety, c.sepal_length, c.rank)

variety,sepal_length,rank
str,f64,i64
"""Setosa""",5.8,1
"""Setosa""",5.7,2
"""Versicolor""",7.0,1
"""Versicolor""",6.9,2
"""Virginica""",7.9,1
"""Virginica""",7.7,2


## `lag` / `lead` — shifted values
Like dplyr, `lag(x)[i] == x[i-1]` and the fill is null by default.

In [14]:
df \
    | head(5) \
    | select(c.sepal_length) \
    | mutate(prev = lag(c.sepal_length), nxt = lead(c.sepal_length))

sepal_length,prev,nxt
f64,f64,f64
5.1,null,4.9
4.9,5.1,4.7
4.7,4.9,4.6
4.6,4.7,5.0
5.0,4.6,null


## Joins
All six dplyr joins are available. Build two small tables to demonstrate.

In [15]:
orders = DataFrame({
    "customer_id": [1, 2, 1, 3, 2],
    "amount": [100, 200, 50, 300, 75],
})
customers = DataFrame({
    "customer_id": [1, 2, 4],
    "name": ["Alice", "Bob", "Dana"],
})
orders

customer_id,amount
i64,i64
1,100
2,200
1,50
3,300
2,75


### `inner_join` — only matching rows

In [16]:
orders | inner_join(customers, by=c.customer_id)

customer_id,amount,name
i64,i64,str
1,100,"""Alice"""
2,200,"""Bob"""
1,50,"""Alice"""
2,75,"""Bob"""


### `left_join` — keep all left rows

In [17]:
orders | left_join(customers, by=c.customer_id)

customer_id,amount,name
i64,i64,str
1,100,"""Alice"""
2,200,"""Bob"""
1,50,"""Alice"""
3,300,null
2,75,"""Bob"""


### `full_join` — keep all rows from both, coalescing the key (like dplyr)

In [18]:
orders | full_join(customers, by=c.customer_id)

customer_id,amount,name
i64,i64,str
1,100,"""Alice"""
2,200,"""Bob"""
1,50,"""Alice"""
3,300,null
2,75,"""Bob"""
4,null,"""Dana"""


### `semi_join` / `anti_join` — filter by presence in another table

In [19]:
# Orders whose customer has a profile vs. orders that don't.
matched = orders | semi_join(customers, by=c.customer_id)
orphans = orders | anti_join(customers, by=c.customer_id)
display(matched)
display(orphans)

customer_id,amount
i64,i64
1,100
2,200
1,50
2,75


customer_id,amount
i64,i64
3,300


### A realistic join pipeline
Total spend per named customer.

In [20]:
orders \
    | inner_join(customers, by=c.customer_id) \
    | group_by(c.name) \
    | summarize(total = c.amount.sum(), orders = n()) \
    | arrange(desc(c.total))

name,total,orders
str,i64,u32
"""Bob""",275,2
"""Alice""",150,2


## `slice_sample` / `sample_n` — random rows
Use `n=` for a fixed count or `prop=` for a fraction.

In [21]:
df | slice_sample(n=5, seed=42)

sepal_length,sepal_width,petal_length,petal_width,variety
f64,f64,f64,f64,str
5.1,3.8,1.6,0.2,"""Setosa"""
6.5,3.0,5.8,2.2,"""Virginica"""
7.7,2.6,6.9,2.3,"""Virginica"""
6.0,2.2,5.0,1.5,"""Virginica"""
6.7,3.0,5.2,2.3,"""Virginica"""


## `pull` — extract a single column as a Series

In [22]:
df | distinct(c.variety) | pull(c.variety)

variety
str
"""Setosa"""
"""Versicolor"""
"""Virginica"""


## Putting it all together
The canonical demo pipeline: Setosa flowers, sepal ratio, top 5.

In [23]:
read_csv("iris.csv") \
    | filter(c.variety == "Setosa") \
    | select(c.sepal_length, c.sepal_width) \
    | mutate(sepal_ratio = c.sepal_length / c.sepal_width) \
    | arrange(desc(c.sepal_ratio)) \
    | head(5)

sepal_length,sepal_width,sepal_ratio
f64,f64,f64
4.5,2.3,1.956522
5.0,3.0,1.666667
4.9,3.0,1.633333
4.8,3.0,1.6
4.8,3.0,1.6
